<a href="https://colab.research.google.com/github/haasinisalihundam-netizen/ML-2520090170/blob/Skill/ML_Skill_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# PLACEMENT DATASET - COMPLETE EDA PDF WITH DOWNLOAD BUTTON
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

from google.colab import files
from IPython.display import display, clear_output
from matplotlib.backends.backend_pdf import PdfPages


# ============================================================
# 2. UPLOAD THE DATASET
# ============================================================

uploaded = files.upload()


# ============================================================
# 3. LOAD THE DATASET
# ============================================================

# Do not change this filename
CSV_PATH = "/content/placement_predict_50k Dataset (1).csv"

df = pd.read_csv(CSV_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)


# ============================================================
# 4. CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()


# ============================================================
# 5. SELECT ONLY THE REQUIRED MODEL-RELATED COLUMNS
# ============================================================

INPUT_COLUMNS = [
    "Stream",
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

TARGET_COLUMN = "PlacementStatus"


# Check whether all required columns are present
required_columns = INPUT_COLUMNS + [TARGET_COLUMN]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"The following required columns are missing: {missing_columns}"
    )


# Create the working DataFrame
data = df[INPUT_COLUMNS + [TARGET_COLUMN]].copy()


# ============================================================
# 6. CREATE A CORRECT NUMERIC PLACEMENT FLAG
# ============================================================

# Use the existing Placed_Flag column when available
if "Placed_Flag" in df.columns:

    data["Placed_Flag"] = pd.to_numeric(
        df["Placed_Flag"],
        errors="coerce"
    )

else:

    # Convert PlacementStatus when it is stored as text
    if data[TARGET_COLUMN].dtype == "object":

        cleaned_status = (
            data[TARGET_COLUMN]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        placement_mapping = {
            "placed": 1,
            "not placed": 0,
            "yes": 1,
            "no": 0,
            "selected": 1,
            "not selected": 0,
            "1": 1,
            "0": 0
        }

        data["Placed_Flag"] = cleaned_status.map(
            placement_mapping
        )

    else:

        # Convert numeric PlacementStatus values
        numeric_status = pd.to_numeric(
            data[TARGET_COLUMN],
            errors="coerce"
        )

        unique_values = sorted(
            numeric_status.dropna().unique()
        )

        if set(unique_values).issubset({0, 1}):

            data["Placed_Flag"] = numeric_status

        elif len(unique_values) == 2:

            # Smaller value becomes 0 and larger value becomes 1
            data["Placed_Flag"] = numeric_status.map({
                unique_values[0]: 0,
                unique_values[1]: 1
            })

        else:

            raise ValueError(
                "PlacementStatus could not be converted into "
                "two placement classes."
            )


# Check whether conversion was successful
if data["Placed_Flag"].isnull().all():
    raise ValueError(
        "PlacementStatus could not be converted into 0 and 1."
    )


# Display the detected placement classes
print("\nPlacement class counts:")

print(
    data["Placed_Flag"]
    .value_counts(dropna=False)
    .sort_index()
)


# ============================================================
# 7. SET GRAPH APPEARANCE
# ============================================================

sns.set_theme(style="whitegrid")


# Continuous columns used for distributions and boxplots
CONTINUOUS_COLUMNS = [
    "CGPA",
    "AttendancePercent",
    "AptitudeTestScore",
    "CodingTestScore",
    "SoftSkillsRating",
    "MockInterviewScore"
]


# Numerical columns used for correlation
CORRELATION_COLUMNS = [
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular",
    "Placed_Flag"
]


# ============================================================
# 8. CREATE THE EDA PDF REPORT
# ============================================================

PDF_NAME = "Placement_EDA_Report.pdf"

PDF_PATH = f"/content/{PDF_NAME}"


with PdfPages(PDF_PATH) as pdf:

    # ========================================================
    # PAGE 1: MISSING VALUES HEATMAP
    # ========================================================

    fig1, ax1 = plt.subplots(figsize=(14, 7))

    sns.heatmap(
        data[INPUT_COLUMNS + ["Placed_Flag"]].isnull(),
        cbar=False,
        yticklabels=False,
        ax=ax1
    )

    ax1.set_title(
        "Missingness Heatmap - Selected Placement Features",
        fontsize=16,
        fontweight="bold"
    )

    ax1.set_xlabel("Selected Columns", fontsize=12)
    ax1.set_ylabel("Student Records", fontsize=12)

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.tight_layout()

    pdf.savefig(
        fig1,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig1)


    # ========================================================
    # PAGE 2: DISTRIBUTION GRAPHS
    # ========================================================

    fig2, axes2 = plt.subplots(
        2,
        3,
        figsize=(17, 10)
    )

    for index, column in enumerate(CONTINUOUS_COLUMNS):

        row, position = divmod(index, 3)

        sns.histplot(
            data=data,
            x=column,
            bins=30,
            kde=True,
            ax=axes2[row, position]
        )

        axes2[row, position].set_title(
            f"Distribution of {column}",
            fontsize=12,
            fontweight="bold"
        )

        axes2[row, position].set_xlabel(column)
        axes2[row, position].set_ylabel(
            "Number of Students"
        )

    fig2.suptitle(
        "Distribution of Important Numerical Features",
        fontsize=17,
        fontweight="bold",
        y=1.02
    )

    plt.tight_layout()

    pdf.savefig(
        fig2,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig2)


    # ========================================================
    # PAGE 3: PLACEMENT RATE BY STREAM
    # ========================================================

    placement_rate = (
        data.dropna(
            subset=["Stream", "Placed_Flag"]
        )
        .groupby("Stream")["Placed_Flag"]
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .reset_index()
    )

    placement_rate.columns = [
        "Stream",
        "Placement Rate (%)"
    ]

    fig3, ax3 = plt.subplots(
        figsize=(12, 7)
    )

    sns.barplot(
        data=placement_rate,
        x="Stream",
        y="Placement Rate (%)",
        hue="Stream",
        legend=False,
        ax=ax3
    )

    ax3.set_title(
        "Placement Rate (%) by Stream",
        fontsize=16,
        fontweight="bold"
    )

    ax3.set_xlabel(
        "Stream",
        fontsize=12
    )

    ax3.set_ylabel(
        "Placement Rate (%)",
        fontsize=12
    )

    ax3.set_ylim(0, 100)

    plt.xticks(
        rotation=30,
        ha="right"
    )

    # Display the percentage above every bar
    for container in ax3.containers:

        ax3.bar_label(
            container,
            fmt="%.2f%%",
            padding=4,
            fontsize=10
        )

    plt.tight_layout()

    pdf.savefig(
        fig3,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig3)


    # ========================================================
    # PAGE 4: CORRELATION HEATMAP
    # ========================================================

    correlation_matrix = (
        data[CORRELATION_COLUMNS]
        .corr(numeric_only=True)
    )

    fig4, ax4 = plt.subplots(
        figsize=(14, 11)
    )

    sns.heatmap(
        correlation_matrix,
        annot=True,
        fmt=".2f",
        center=0,
        linewidths=0.5,
        ax=ax4
    )

    ax4.set_title(
        "Correlation Heatmap - Selected Numerical Features",
        fontsize=16,
        fontweight="bold"
    )

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.yticks(rotation=0)

    plt.tight_layout()

    pdf.savefig(
        fig4,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig4)


    # ========================================================
    # PAGE 5: BOXPLOTS
    # ========================================================

    fig5, axes5 = plt.subplots(
        2,
        3,
        figsize=(17, 10)
    )

    for index, column in enumerate(CONTINUOUS_COLUMNS):

        row, position = divmod(index, 3)

        sns.boxplot(
            data=data,
            y=column,
            ax=axes5[row, position]
        )

        axes5[row, position].set_title(
            f"Boxplot of {column}",
            fontsize=12,
            fontweight="bold"
        )

        axes5[row, position].set_xlabel("")
        axes5[row, position].set_ylabel(column)

    fig5.suptitle(
        "Outlier Analysis of Important Numerical Features",
        fontsize=17,
        fontweight="bold",
        y=1.02
    )

    plt.tight_layout()

    pdf.savefig(
        fig5,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig5)


# ============================================================
# 9. CONFIRM THAT THE REPORT WAS CREATED
# ============================================================

if os.path.exists(PDF_PATH):

    print("\nEDA report created successfully.")
    print("Report name:", PDF_NAME)
    print("Number of pages: 5")

else:

    raise FileNotFoundError(
        "The EDA PDF report was not created."
    )


# ============================================================
# 10. CREATE THE DOWNLOAD BUTTON
# ============================================================

download_button = widgets.Button(
    description="Download EDA Report",
    button_style="success",
    icon="download",
    tooltip="Click to download the Placement EDA PDF",
    layout=widgets.Layout(
        width="260px",
        height="48px"
    )
)


download_output = widgets.Output()


def download_eda_report(button):

    with download_output:

        clear_output()

        if os.path.exists(PDF_PATH):

            print("Preparing your EDA report for download...")

            files.download(PDF_PATH)

        else:

            print("The EDA report could not be found.")


download_button.on_click(download_eda_report)


print("\nClick the button below to download the report:")

display(download_button)
display(download_output)